# Análise Exploratória de Dados (AED)

## 1. Apresentação da análise

### 1.1. Problema analítico
A *Online Shop* trata toda a base de clientes de forma uniforme nas campanhas de marketing, sem inteligência sobre o comportamento de recompra. Isso gera dois custos: clientes fiéis recebem promoções desnecessárias (corroendo margem e elevando o *opt-out*) e orçamento é desperdiçado com clientes já inativos. O objetivo desta AED é **diagnosticar padrões de engajamento, inatividade e geração de receita** que apoiem decisões de marketing mais segmentadas.

### 1.2. Perguntas investigáveis
- **Q1.** Como o tempo de inatividade desde a última compra distribui os clientes e quem compõe o maior risco?
- **Q2.** Qual é o volume de recompra necessário para que um cliente faça parte dos mais fiéis à loja?
- **Q3.** Qual é a relação entre o comportamento de compra dos clientes e o retorno financeiro que eles geram?

### 1.3. Dataset utilizado
*Online Shop 2024 Dataset* (Kaggle), já processado pelo pipeline do projeto até a **camada gold** (arquitetura *medallion*: bronze → silver → gold). A camada gold consolida o histórico transacional em um perfil **RFM** (Recência, Frequência, Valor) por cliente.

### 1.4. Unidade de análise
O **cliente** (`customer_id`). Cada linha representa um cliente único; pedidos, itens e pagamentos já foram agregados a esse nível.

### 1.5. Versão dos dados
Camada **gold** do pipeline (dados de 2024). Esta AED não altera os dados; consome a versão consolidada.

### 1.6. Escopo da análise exploratória
Análise **exploratória e descritiva**: descrição das variáveis, distribuições, dispersão, outliers, relações iniciais e padrões. Não inclui modelagem preditiva nem inferência causal; conclusões definitivas ficam para etapas posteriores do pipeline.

## 2. Visão geral do dataset

### 2.1. Importação dos pacotes

- `pandas`: manipulação e análise de dados tabulares;
- `numpy`: operações numéricas auxiliares;
- `pathlib`: path do dataset;
- `seaborn`: carregamento do dataset e visualização estatística;
- `matplotlib.pyplot`: configuração e exibição de gráficos.
- `pyarrow`: leitura/escrita de arquivos no formato Parquet (usado na exportação da camada Silver);

In [ ]:
%pip install numpy pandas matplotlib seaborn scipy pyarrow

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

# Configurações visuais
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 2)

# Caminho dataset
PATH_GOLD   = Path('../datasets/dataset_gold/')

print(f'Gold:   {PATH_GOLD.resolve()}')

### 2.2. Carregamento do dataset

Cada linha representa um cliente.

In [ ]:
df = pd.read_parquet(PATH_GOLD / "dataset_gold.parquet")

df.head()

### 2.3. Dimensão do dataset

In [ ]:
linhas, colunas = df.shape

print(f"Número de linhas: {linhas}")
print(f"Número de colunas: {colunas}")

A base reúne **10.000 clientes** descritos por **14 atributos**. A granularidade (um cliente por linha) é coerente com a unidade de análise definida e com as perguntas Q1–Q3, todas formuladas no nível do cliente.

### 2.4. Estrutura das variáveis


In [ ]:
df.info()

Há três blocos de variáveis: identificador (`customer_id`), datas (`primeira_compra`, `ultima_compra`) e as métricas RFM derivadas (recência, frequência, valor e seus *scores*). Os tipos estão coerentes datas como `datetime`, métricas como numéricas, `is_recomprador` como booleano e `segmento_cliente` como texto.

### 2.5. Valores ausentes

In [ ]:
missing = df.isnull().sum()

missing

Não foram identificados valores ausentes.

### 2.6. Valores duplicados

In [ ]:
df.duplicated().sum()

Não foram identificados registros duplicados.

### 2.7. Identificação de variáveis numéricas e categóricas

In [ ]:
# Papéis analíticos explícitos (usados no restante do notebook)
metricas_numericas = ["recencia_dias", "frequencia", "valor_monetario", "ticket_medio"]
scores_ordinais    = ["score_recencia", "score_frequencia", "score_monetario",
                      "score_frequencia_monetario", "score_rfm"]
categoricas_aed    = scores_ordinais + ["is_recomprador", "segmento_cliente"]

print("Métricas numéricas:", metricas_numericas)
print("Scores ordinais   :", scores_ordinais)
print("Categóricas (AED) :", categoricas_aed)

### 2.8. Variáveis selecionadas para a AED e justificativa

Organizamos as variáveis por papel analítico:

| Papel | Variáveis | Uso na AED |
|---|---|---|
| **Métricas numéricas (RFM)** | `recencia_dias`, `frequencia`, `valor_monetario`, `ticket_medio` | Estatística descritiva, distribuições, dispersão, outliers, correlação |
| **Scores ordinais (1–4)** | `score_recencia`, `score_frequencia`, `score_monetario`, `score_frequencia_monetario`, `score_rfm` | Tratados como **categóricos ordinais**: frequências e contraste entre grupos |
| **Indicador / segmento** | `is_recomprador`, `segmento_cliente` | Comparações entre grupos e proporções condicionais |
| **Excluídas da análise direta** | `customer_id`, `primeira_compra`, `ultima_compra` | `customer_id` é apenas identificador; as datas brutas servem para derivar recência e tendência temporal, não para estatística descritiva |

**Decisão sobre os `score_*`:** por serem notas de 1 a 4 (escala ordinal), são analisados como categóricos ordinais (frequências, *countplots*), e não com média/desvio como se fossem contínuos. *(Pendência confirmada com o grupo; ajuste aqui se a convenção do projeto for outra.)*

## 3. Descrição estatística

### 3.1. Resumo estatístico geral das variáveis numéricas

- `count`: quantidade de valores não ausentes;
- `mean`: média;
- `std`: desvio-padrão;
- `min`: menor valor;
- `25%`: primeiro quartil;
- `50%`: mediana;
- `75%`: terceiro quartil;
- `max`: maior valor.

In [ ]:
df[metricas_numericas].describe()

### 3.2. Média e mediana das variáveis numéricas

In [ ]:
media_mediana = pd.DataFrame({
    "media": df[metricas_numericas].mean(),
    "mediana": df[metricas_numericas].median()
})
media_mediana["dif_media_mediana"] = media_mediana["media"] - media_mediana["mediana"]
media_mediana

### 3.3. Mínimo, máximo e amplitude inicial das variáveis numéricas


In [ ]:
limites = pd.DataFrame({
    "minimo": df[metricas_numericas].min(),
    "maximo": df[metricas_numericas].max()
})
limites["amplitude"] = limites["maximo"] - limites["minimo"]
limites

### 3.4. Quartis das variáveis numéricas

In [ ]:
quartis = df[metricas_numericas].quantile([0.25, 0.50, 0.75])
quartis.index = ["Q1_25%", "Q2_50%_mediana", "Q3_75%"]
quartis

### 3.5. Frequências absolutas das variáveis categóricas

In [ ]:
for coluna in categoricas_aed:
    print(f"\nFrequência absoluta — {coluna}")
    display(df[coluna].value_counts(dropna=False).to_frame(name="frequencia"))

### 3.6. Frequências relativas das variáveis categóricas


In [ ]:
for coluna in categoricas_aed:
    print(f"\nProporção — {coluna}")
    prop = df[coluna].value_counts(normalize=True, dropna=False).to_frame(name="proporcao")
    prop["percentual"] = prop["proporcao"] * 100
    display(prop)

As proporções mostram a composição da base: a parcela de **recompradores** (`is_recomprador = True`) e a distribuição entre **segmentos**. Scores concentrados nas notas baixas (1–2) indicam uma base predominantemente de baixo engajamento — coerente com Q1 e Q2.

### 3.7. Resumo combinado das variáveis categóricas

In [ ]:
resumo_cat = []
for coluna in categoricas_aed:
    freq = df[coluna].value_counts(dropna=False)
    resumo_cat.append({
        "variavel": coluna,
        "n_categorias": df[coluna].nunique(dropna=False),
        "categoria_predominante": freq.index[0],
        "freq_predominante": int(freq.iloc[0]),
        "prop_predominante_%": round(freq.iloc[0] / len(df) * 100, 1),
    })
pd.DataFrame(resumo_cat)

A tabela sintetiza, por variável categórica, a categoria predominante e seu peso relativo — útil para identificar rapidamente onde a base se concentra e quais categorias são raras.

## 4. Distribuições, dispersão e outliers

Aprofundamos a forma das quatro métricas numéricas: como os valores se distribuem, quanto variam e quais extremos merecem sinalização.

### 4.1. Histogramas das métricas numéricas

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, var in zip(axes.flat, metricas_numericas):
    sns.histplot(data=df, x=var, bins=40, kde=True, ax=ax)
    ax.set_title(f"Distribuição — {var}")
fig.tight_layout()
plt.show()

`valor_monetario`, `ticket_medio` e `frequencia` apresentam **assimetria à direita**: massa concentrada em valores baixos e cauda longa de poucos clientes muito acima da média. `recencia_dias` é mais espalhada, com tendência de concentração nos clientes recentes e uma cauda de inativos — a base do risco investigado em Q1.

### 4.2. Boxplots das métricas numéricas

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 5))
for ax, var in zip(axes, metricas_numericas):
    sns.boxplot(data=df, y=var, ax=ax)
    ax.set_title(var)
fig.tight_layout()
plt.show()

Os boxplots tornam visíveis os quartis e os pontos além dos *whiskers*. Em valor e frequência há muitos pontos acima do limite superior — coerente com a cauda longa observada nos histogramas. Esses pontos são **sinalizados**, não removidos.

### 4.3. Medidas de dispersão comparadas (desvio-padrão e IQR)

In [ ]:
disp = []
for var in metricas_numericas:
    s = df[var]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    disp.append({
        "variavel": var,
        "desvio_padrao": s.std(),
        "amplitude": s.max() - s.min(),
        "IQR": q3 - q1,
    })
pd.DataFrame(disp)

Amplitude, desvio-padrão e IQR captam aspectos diferentes da variação. O contraste entre uma amplitude grande e um IQR pequeno (típico em valor e frequência) confirma que **a dispersão total é dominada por poucos extremos**, enquanto a metade central dos clientes é relativamente homogênea.

### 4.4. Assimetria (skewness)

In [ ]:
assimetria = df[metricas_numericas].apply(lambda s: stats.skew(s, bias=False))
assimetria.sort_values(ascending=False).to_frame(name="skewness")

Valores de assimetria claramente positivos (> 0) confirmam a cauda à direita em valor, ticket e frequência. Em variáveis assimétricas, a **mediana** descreve melhor o cliente típico do que a média.

### 4.5. Sinalização de outliers pelo critério do IQR

In [ ]:
def outliers_iqr(serie):
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    li, ls = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (serie < li) | (serie > ls)
    return li, ls, mask.sum(), mask.mean() * 100

linhas_out = []
for var in metricas_numericas:
    li, ls, n, pct = outliers_iqr(df[var])
    linhas_out.append({"variavel": var, "limite_inf": round(li, 2),
                       "limite_sup": round(ls, 2), "n_outliers": n,
                       "pct_outliers_%": round(pct, 2)})
pd.DataFrame(linhas_out)

O critério `Q1 − 1,5·IQR` / `Q3 + 1,5·IQR` (o mesmo adotado no material da disciplina) é uma **triagem**, não uma regra de remoção. Os outliers superiores de `valor_monetario` provavelmente correspondem a clientes de altíssimo valor — casos raros **relevantes** para o negócio (Q3), e não erros. A decisão de tratamento pertence à etapa de modelagem, com base no dicionário de dados e nas regras de negócio.

### 4.6. Distribuição da frequência de compra

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.countplot(data=df, x="frequencia", ax=ax,
              order=sorted(df["frequencia"].unique()))
ax.set_title("Número de pedidos por cliente")
ax.set_xlabel("frequencia (nº de pedidos)")
ax.set_ylabel("nº de clientes")
plt.show()

A frequência é uma variável discreta fortemente concentrada em **1 pedido**, com queda acentuada para valores maiores. Esse formato é a evidência central para Q2: a fidelidade (recompra) é um comportamento minoritário na base.

### 4.7. Interpretação geral da seção
As métricas de valor e frequência são **assimétricas à direita**, com poucos clientes de alto valor sustentando a cauda; a recência distingue clientes recentes de uma cauda de inativos. Os outliers são, em boa parte, **casos raros relevantes** (alto valor) e devem ser preservados na exploração.

## 5. Relações iniciais entre variáveis

Saímos da análise univariada para investigar como as variáveis se comportam em conjunto.

### 5.1. Categórica × categórica: recompra por segmento

In [ ]:
tab = pd.crosstab(df["segmento_cliente"], df["is_recomprador"])
display(tab)

prop_linha = pd.crosstab(df["segmento_cliente"], df["is_recomprador"], normalize="index") * 100
print("\nProporção por linha (% dentro de cada segmento):")
display(prop_linha.round(1))

In [ ]:
prop_linha.plot(kind="bar", stacked=True, figsize=(9, 5))
plt.title("Composição de recompradores dentro de cada segmento")
plt.ylabel("%")
plt.legend(title="is_recomprador", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

A proporção de recompradores muda conforme o segmento — indício exploratório de **associação** entre `segmento_cliente` e `is_recomprador`. Segmentos de maior valor concentram recompradores; segmentos de risco/inativos concentram clientes de compra única.

### 5.2. Numérica × numérica: dispersões

In [ ]:
pares = [("frequencia", "valor_monetario"),
         ("recencia_dias", "valor_monetario"),
         ("frequencia", "ticket_medio")]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (x, y) in zip(axes, pares):
    sns.scatterplot(data=df, x=x, y=y, alpha=0.25, s=15, ax=ax)
    ax.set_title(f"{x} × {y}")
fig.tight_layout()
plt.show()

`frequencia × valor_monetario` mostra relação positiva (comprar mais tende a somar mais receita). Já `frequencia × ticket_medio` aparece **sem relação clara**: clientes que compram com mais frequência não gastam necessariamente mais por pedido — leitura diretamente ligada a Q3-H2.

### 5.3. Correlação linear entre as métricas

In [ ]:
corr = df[metricas_numericas].corr()
display(corr.round(2))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            vmin=-1, vmax=1, ax=ax)
ax.set_title("Matriz de correlação — métricas numéricas")
plt.tight_layout()
plt.show()

### 5.4. Categórica × numérica: métricas por segmento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ordem = df.groupby("segmento_cliente")["valor_monetario"].median().sort_values().index
sns.boxplot(data=df, x="segmento_cliente", y="valor_monetario", order=ordem, ax=axes[0])
axes[0].set_title("Valor monetário por segmento")
axes[0].tick_params(axis="x", rotation=30)
sns.boxplot(data=df, x="segmento_cliente", y="recencia_dias", order=ordem, ax=axes[1])
axes[1].set_title("Recência por segmento")
axes[1].tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

Há **contraste claro entre grupos**: segmentos de maior valor exibem distribuições de `valor_monetario` deslocadas para cima e recência menor (clientes mais recentes), enquanto segmentos inativos/de risco mostram o oposto. Isso sustenta o uso do segmento como eixo de decisão de marketing.

### 5.5. Média geral versus médias por grupo

In [ ]:
geral = df["valor_monetario"].mean()
por_seg = df.groupby("segmento_cliente")["valor_monetario"].agg(["mean", "median", "count"])
por_seg["mean_vs_geral"] = por_seg["mean"] - geral
print(f"Média geral de valor_monetario: {geral:,.2f}")
por_seg.round(2)

A média geral esconde diferenças relevantes: ao segmentar, alguns grupos ficam muito acima e outros muito abaixo da média global. Resumos únicos sobre toda a base podem, portanto, **mascarar os subgrupos** que mais importam para a estratégia.

### 5.6. Interpretação geral da seção
As relações observadas convergem: valor caminha com frequência, mas **não com o ticket médio**; recência e valor separam segmentos; e a recompra concentra-se nos segmentos de maior valor. São indícios exploratórios alinhados a Q3, a serem testados formalmente adiante.

## 6. Padrões, tendências e anomalias exploratórias

### 6.1. Concentração de receita (Q3)

In [ ]:
v = df["valor_monetario"].sort_values(ascending=False).reset_index(drop=True)
receita_acum = v.cumsum() / v.sum()
pct_clientes = (np.arange(1, len(v) + 1)) / len(v)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pct_clientes * 100, receita_acum * 100)
ax.set_xlabel("% de clientes (ordenados por valor, do maior para o menor)")
ax.set_ylabel("% acumulado da receita")
ax.set_title("Curva de concentração de receita")
for marca in [0.10, 0.20]:
    idx = int(marca * len(v)) - 1
    ax.axvline(marca * 100, color="gray", ls="--", lw=0.8)
    ax.annotate(f"top {int(marca*100)}% → {receita_acum.iloc[idx]*100:.0f}% da receita",
                (marca * 100, receita_acum.iloc[idx] * 100))
plt.tight_layout()
plt.show()

A curva mostra **concentração de receita**: uma fração reduzida de clientes responde por boa parte do faturamento (Q3-H1). O grau exato (ex.: quanto vem do top 20%) deve ser lido diretamente do gráfico/dados reais; a forma côncava acentuada é o que caracteriza a concentração, sem assumir a regra "80/20" a priori.

### 6.2. Contraste entre segmentos (medianas)

In [ ]:
perfil = df.groupby("segmento_cliente").agg(
    n_clientes=("customer_id", "count"),
    recencia_mediana=("recencia_dias", "median"),
    frequencia_mediana=("frequencia", "median"),
    valor_mediano=("valor_monetario", "median"),
    ticket_mediano=("ticket_medio", "median"),
    receita_total=("valor_monetario", "sum"),
).sort_values("valor_mediano", ascending=False)
perfil["%_receita_total"] = (perfil["receita_total"] / df["valor_monetario"].sum() * 100).round(1)
perfil.round(2)

O perfil por segmento resume o diagnóstico: poucos segmentos de alto valor concentram a maior fatia da `receita_total`, enquanto segmentos numerosos de baixo valor contribuem pouco. É a tradução operacional do padrão de concentração para a ação de marketing.

### 6.3. Tendência temporal: aquisição de clientes ao longo de 2024

In [ ]:
aquisicao = (df.set_index("primeira_compra")
               .resample("ME")["customer_id"].count())
fig, ax = plt.subplots(figsize=(10, 4.5))
aquisicao.plot(marker="o", ax=ax)
ax.set_title("Clientes adquiridos por mês (data da primeira compra)")
ax.set_xlabel("mês")
ax.set_ylabel("nº de novos clientes")
plt.tight_layout()
plt.show()

A série de **primeiras compras** descreve o ritmo de aquisição ao longo do ano. Variações entre meses sugerem sazonalidade na entrada de clientes (relacionada a Q1-H2); como tendência exploratória, deve ser confirmada com mais períodos antes de qualquer conclusão.

### 6.4. Anomalias exploratórias: inativos de alto valor

In [ ]:
lim_recencia = df["recencia_dias"].quantile(0.75)   # 25% mais inativos
lim_valor = df["valor_monetario"].quantile(0.90)     # 10% mais valiosos
anomalos = df[(df["recencia_dias"] >= lim_recencia) & (df["valor_monetario"] >= lim_valor)]
print(f"Clientes com recência alta (>= P75={lim_recencia:.0f} dias) "
      f"E valor alto (>= P90={lim_valor:,.0f}): {len(anomalos)}")
anomalos[["customer_id", "recencia_dias", "frequencia",
          "valor_monetario", "segmento_cliente"]].head(10)

Esse grupo — clientes valiosos que **pararam de comprar** — é um comportamento que merece investigação: representa receita em risco e é candidato natural a campanhas de reativação. O objetivo aqui é sinalizar o padrão, não corrigi-lo automaticamente.

### 6.5. Interpretação geral da seção
Três padrões se destacam: **concentração de receita** em poucos clientes (Q3), **contraste estrutural entre segmentos** (Q1) e a **predominância de compradores únicos** (Q2). A aquisição apresenta variação temporal e há um subgrupo relevante de inativos de alto valor.

## 7. Formulação de hipóteses exploratórias

Cada hipótese está vinculada a uma evidência do notebook e a uma pergunta investigável, é específica e plausível, e evita afirmações causais. São direções de investigação para etapas posteriores — não conclusões.

**H1 (Q2).** *A maioria dos clientes realiza apenas um pedido.*
Evidência: distribuição de `frequencia` fortemente concentrada em 1 (Seções 3.6 e 4.6).

**H2 (Q3).** *Uma fração reduzida de clientes concentra a maior parte da receita.*
Evidência: curva de concentração de receita acentuadamente côncava e `%_receita_total` por segmento (Seções 6.1 e 6.2).

**H3 (Q3).** *Maior frequência de compra não está associada a maior ticket médio.*
Evidência: dispersão `frequencia × ticket_medio` sem padrão e correlação próxima de zero (Seções 5.2 e 5.3).

**H4 (Q1).** *Clientes de alta recência (inativos) concentram-se em segmentos de baixo valor, mas existe um subgrupo de inativos de alto valor.*
Evidência: boxplots de `recencia_dias`/`valor_monetario` por segmento e o grupo anômalo identificado (Seções 5.4 e 6.4).

**H5 (Q1).** *O volume de aquisição de clientes varia ao longo dos meses de 2024.*
Evidência: série mensal de primeiras compras com oscilação entre períodos (Seção 6.3).

## 8. Síntese final

A AED da camada gold (10.000 clientes, perfil RFM) indica uma base **assimétrica**: a maioria compra uma única vez e contribui pouco, enquanto poucos clientes de alto valor sustentam grande parte da receita (**Q3**). A recência separa clientes recentes de uma cauda de inativos, e os segmentos diferem de forma marcante em valor e recência (**Q1**); a recompra é minoritária e concentrada nos segmentos de maior valor (**Q2**). Frequência e valor caminham juntos, mas frequência e ticket médio são praticamente independentes.

Esses achados são **exploratórios**: orientam as hipóteses H1–H5 e a etapa de segmentação/modelagem, sem estabelecer relações causais nem conclusões definitivas a partir das evidências aqui observadas.